# Notebook 07 — The Quantum Fourier Transform

**Prerequisites:** Notebooks 01-06. Familiarity with controlled-phase gates and entanglement.

The **Quantum Fourier Transform (QFT)** is the quantum analog of the classical Discrete Fourier Transform (DFT). While the classical DFT on $N = 2^n$ amplitudes requires $O(N \log N)$ operations (via the FFT), the QFT accomplishes the same transformation using only $O(n^2)$ quantum gates — an **exponential speedup**.

The QFT is not a standalone algorithm; it is a subroutine that appears at the heart of many of quantum computing's most powerful algorithms:

- **Shor's algorithm** for integer factoring (via order finding)
- **Quantum Phase Estimation (QPE)** — covered in the next notebook
- **Quantum counting** and **HHL** (linear systems solver)

By the end of this notebook you will be able to:

1. **Describe** the QFT circuit structure and its relation to the classical DFT.
2. **Implement** a QFT manually from H and controlled-phase gates.
3. **Verify** that QFT followed by inverse QFT is the identity.
4. **Explain** the phase kickback mechanism.

In this notebook we will:

1. Understand what the QFT does mathematically.
2. Build a QFT circuit from scratch using H and controlled-phase gates.
3. Use Goqu's built-in `qpe.QFT` and `qpe.InverseQFT`.
4. Verify that QFT followed by inverse QFT is the identity.
5. Observe the QFT's effect on different input states.
6. Preview **phase kickback**, the mechanism behind QPE.
7. Explore circuit composition utilities: `ir.Inverse`, `ir.Repeat`, `Compose`, and `ComposeInverse`.

In [1]:
import (
	"fmt"
	"math"
	"math/cmplx"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/qpe"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## What Is the QFT?

The Discrete Fourier Transform maps a vector of $N$ complex amplitudes from the **computational basis** to the **frequency basis**. On $n$ qubits ($N = 2^n$), the QFT acts as:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i \, jk / N} |k\rangle$$

Key observations:

- The QFT maps **computational basis states** to states with well-defined **phase structure**.
- Applied to $|0\rangle$, every output amplitude has the same phase ($e^{0} = 1$), giving the **uniform superposition**.
- Applied to $|j\rangle$ with $j > 0$, the amplitudes acquire phases that increase linearly with $k$ — a frequency encoding.
- The QFT is a **unitary operation** — it has an inverse, the inverse QFT.

### QFT Circuit Structure

The QFT on $n$ qubits decomposes into:

1. For each qubit $i$ (from 0 to $n-1$):
   - Apply **H** to qubit $i$.
   - For each subsequent qubit $j > i$: apply a **controlled-phase rotation** $\text{CP}(\pi / 2^{j-i})$ between qubits $i$ and $j$.
2. **SWAP** qubits to reverse their order (standard QFT output convention).

This requires $n$ Hadamard gates and $n(n-1)/2$ controlled-phase gates — a total of $O(n^2)$ gates.

In [2]:
%%
// Build a 3-qubit QFT manually using H and CP gates.
b := builder.New("qft-manual", 3)

// Qubit 0: H, then CP(pi/2) controlled by q1, CP(pi/4) controlled by q2
b.H(0)
b.Apply(gate.CP(math.Pi/2), 1, 0)
b.Apply(gate.CP(math.Pi/4), 2, 0)

// Qubit 1: H, then CP(pi/2) controlled by q2
b.H(1)
b.Apply(gate.CP(math.Pi/2), 2, 1)

// Qubit 2: H
b.H(2)

// SWAP q0 and q2 to match standard QFT output ordering
b.SWAP(0, 2)

manualQFT, err := b.Build()
if err != nil {
	panic(err)
}

fmt.Println("Manual 3-qubit QFT circuit:")
gonbui.DisplayHTML(draw.SVG(manualQFT))

Manual 3-qubit QFT circuit:


q0 
 
 q1 
 
 q2 
 
 H 
 
 
 P(pi/2) 
 
 
 P(pi/4) 
 H 
 
 
 P(pi/2) 
 H

The circuit above shows the characteristic QFT pattern:

- **Hadamard gates** create superposition on each qubit.
- **Controlled-phase gates** encode the frequency relationships between qubits. The angle $\pi / 2^{j-i}$ gets smaller as qubits get farther apart.
- **SWAP gates** at the end reverse the qubit ordering to match the standard DFT convention.

## Using Goqu's Built-In QFT

Goqu provides `qpe.QFT(n)` which constructs the same circuit automatically. Let's compare.

In [3]:
%%
// Use the built-in QFT
builtinQFT, err := qpe.QFT(3)
if err != nil {
	panic(err)
}

fmt.Println("Built-in 3-qubit QFT circuit:")
gonbui.DisplayHTML(draw.SVG(builtinQFT))

Built-in 3-qubit QFT circuit:


q0 
 
 q1 
 
 q2 
 
 H 
 
 
 P(pi/2) 
 
 
 P(pi/4) 
 H 
 
 
 P(pi/2) 
 H

In [4]:
%%
// Rebuild manualQFT from cell above (needed because %% cells have isolated scope)
bManual := builder.New("qft-manual", 3)
bManual.H(0)
bManual.Apply(gate.CP(math.Pi/2), 1, 0)
bManual.Apply(gate.CP(math.Pi/4), 2, 0)
bManual.H(1)
bManual.Apply(gate.CP(math.Pi/2), 2, 1)
bManual.H(2)
bManual.SWAP(0, 2)
manualQFT, _ := bManual.Build()

// Rebuild builtinQFT (needed because %% cells have isolated scope)
builtinQFT, _ := qpe.QFT(3)

// Verify both circuits produce the same statevector on a test input.
// We'll apply each to |101> (decimal 5) and compare.
prepCircuit := func(name string) *ir.Circuit {
	c, _ := builder.New(name, 3).X(0).X(2).Build()
	return c
}

sim1 := statevector.New(3)
sim1.Evolve(prepCircuit("prep1"))
sim1.Apply(manualQFT)
sv1 := sim1.StateVector()

sim2 := statevector.New(3)
sim2.Evolve(prepCircuit("prep2"))
sim2.Apply(builtinQFT)
sv2 := sim2.StateVector()

fmt.Println("Statevectors match after QFT on |101>:")
match := true
for i := range sv1 {
	diff := cmplx.Abs(sv1[i] - sv2[i])
	if diff > 1e-10 {
		match = false
		fmt.Printf("  MISMATCH at |%03b>: manual=%.4f, builtin=%.4f\n", i, sv1[i], sv2[i])
	}
}
if match {
	fmt.Println("  All amplitudes agree to within 1e-10.")
}

Statevectors match after QFT on |101>:
  All amplitudes agree to within 1e-10.


## The Inverse QFT

Since the QFT is a unitary transformation, it has a well-defined inverse. The **inverse QFT** reverses the frequency-domain encoding and maps back to the computational basis:

$$\text{QFT}^{-1} \cdot \text{QFT} = I$$

The inverse QFT circuit reverses the gate order and conjugates all rotation angles. In Goqu, `qpe.InverseQFT(n)` builds this circuit directly.

Let's verify the identity property: applying QFT then inverse QFT should return the original state.

In [5]:
%%
// Rebuild builtinQFT (needed because %% cells have isolated scope)
builtinQFT, _ := qpe.QFT(3)

// Build the inverse QFT
invQFT, err := qpe.InverseQFT(3)
if err != nil {
	panic(err)
}

fmt.Println("Inverse QFT circuit:")
gonbui.DisplayHTML(draw.SVG(invQFT))

// Prepare state |110> (decimal 6)
prep, _ := builder.New("prep", 3).X(1).X(2).Build()

// Evolve: prep -> QFT -> inverse QFT
sim := statevector.New(3)
sim.Evolve(prep)
svOriginal := sim.StateVector()

sim.Apply(builtinQFT)
svAfterQFT := sim.StateVector()

sim.Apply(invQFT)
svRoundTrip := sim.StateVector()

fmt.Println("Original state |110>:")
for i, amp := range svOriginal {
	if cmplx.Abs(amp) > 1e-10 {
		fmt.Printf("  |%03b> : %.4f\n", i, amp)
	}
}

fmt.Println("\nAfter QFT (frequency domain):")
for i, amp := range svAfterQFT {
	if cmplx.Abs(amp) > 1e-10 {
		fmt.Printf("  |%03b> : %+.4f %+.4fi  (|amp|=%.4f)\n", i, real(amp), imag(amp), cmplx.Abs(amp))
	}
}

fmt.Println("\nAfter QFT then inverse QFT (round-trip):")
for i, amp := range svRoundTrip {
	if cmplx.Abs(amp) > 1e-10 {
		fmt.Printf("  |%03b> : %.4f\n", i, amp)
	}
}

// Verify the round-trip reproduces the original
match := true
for i := range svOriginal {
	if cmplx.Abs(svOriginal[i]-svRoundTrip[i]) > 1e-10 {
		match = false
		break
	}
}
fmt.Printf("\nRound-trip matches original: %v\n", match)

Inverse QFT circuit:
Original state |110>:
  |110> : (1.0000+0.0000i)

After QFT (frequency domain):
  |000> : +0.3536 +0.0000i  (|amp|=0.3536)
  |001> : -0.3536 +0.0000i  (|amp|=0.3536)
  |010> : -0.0000 -0.3536i  (|amp|=0.3536)
  |011> : +0.0000 +0.3536i  (|amp|=0.3536)
  |100> : -0.2500 +0.2500i  (|amp|=0.3536)
  |101> : +0.2500 -0.2500i  (|amp|=0.3536)
  |110> : +0.2500 +0.2500i  (|amp|=0.3536)
  |111> : -0.2500 -0.2500i  (|amp|=0.3536)

After QFT then inverse QFT (round-trip):
  |110> : (1.0000+0.0000i)

Round-trip matches original: true


q0 
 
 q1 
 
 q2 
 
 
 
 
 H 
 
 
 P(-pi/2) 
 H 
 
 
 P(-pi/4) 
 
 
 P(-pi/2) 
 H

## QFT on Specific States

To build intuition, let's observe the QFT's output on several computational basis states.

Recall the QFT formula: $\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k} e^{2\pi i jk/N} |k\rangle$

- **$|000\rangle$ ($j=0$):** All phases are $e^{0} = 1$, producing a uniform superposition with equal real amplitudes.
- **$|001\rangle$ and $|010\rangle$:** The QFT encodes different input values as distinct phase patterns in the output. The exact phase distribution depends on the qubit ordering convention (the QFT circuit processes qubit 0 first, treating it as the most significant bit internally).

The important point: each input state produces a **unique phase fingerprint** that the inverse QFT can decode back to the original state.

In [6]:
%%
// Apply QFT to |000>, |001>, and |010> and display the results.
qftCirc, _ := qpe.QFT(3)
N := 8.0 // 2^3

testStates := []struct {
	name  string
	setup func(b *builder.Builder)
}{
	{"000", func(b *builder.Builder) {}},           // |000> = j=0
	{"001", func(b *builder.Builder) { b.X(0) }},   // |001> = j=1
	{"010", func(b *builder.Builder) { b.X(1) }},   // |010> = j=2
}

for _, ts := range testStates {
	fmt.Printf("=== QFT|%s> ===\n", ts.name)

	b := builder.New("prep-"+ts.name, 3)
	ts.setup(b)
	prep, _ := b.Build()

	sim := statevector.New(3)
	sim.Evolve(prep)
	sim.Apply(qftCirc)
	sv := sim.StateVector()

	for k, amp := range sv {
		phase := cmplx.Phase(amp)
		if cmplx.Abs(amp) < 1e-10 {
			phase = 0
		}
		fmt.Printf("  |%03b> : amp = %+.4f %+.4fi  |amp| = %.4f  phase = %.3f rad\n",
			k, real(amp), imag(amp), cmplx.Abs(amp), phase)
	}
	fmt.Printf("  Expected: uniform amplitudes 1/sqrt(%v) = %.4f with linearly increasing phases\n\n",
		N, 1/math.Sqrt(N))
}

=== QFT|000> ===
  |000> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |001> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |010> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |011> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |100> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |101> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |110> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |111> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  Expected: uniform amplitudes 1/sqrt(8) = 0.3536 with linearly increasing phases

=== QFT|001> ===
  |000> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |001> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |010> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |011> : amp = +0.3536 +0.0000i  |amp| = 0.3536  phase = 0.000 rad
  |100> : amp = -0.3536 +0.0000i  |amp| = 0.3536  phase = 3.142 rad

Notice the pattern:

- **QFT|000>**: All amplitudes are $1/\sqrt{8} \approx 0.3536$ with zero phase — a uniform superposition. This is the "zero frequency" — no oscillation.
- **QFT|001>** and **QFT|010>**: All amplitudes have the same magnitude $1/\sqrt{8}$, but with distinct phase patterns. Each input state produces a unique phase fingerprint.
- The amplitudes are always **uniform** ($1/\sqrt{N}$) — only the phases differ between inputs.

The QFT encodes the input value $j$ as a **phase structure** in the output amplitudes. This is the key to algorithms like QPE, where we use the inverse QFT to extract a phase that has been encoded in a quantum register.

**Note on qubit ordering:** The QFT circuit processes qubit 0 first, effectively treating it as the most significant bit within the transform. Combined with the bitstring convention ($|q_2\,q_1\,q_0\rangle$, qubit 0 rightmost), the phase patterns may not appear as simple linear progressions when listed by state index. The important property is that QFT followed by inverse QFT is the identity — the encoding is perfectly reversible regardless of convention.

## Phase Kickback

**Phase kickback** is the mechanism that makes QPE work. The idea is:

1. Prepare a control qubit in superposition: $|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$.
2. Apply a controlled-U gate where the target is an **eigenstate** of U with eigenvalue $e^{i\phi}$.
3. The eigenvalue phase "kicks back" onto the control qubit: $(|0\rangle + e^{i\phi}|1\rangle)/\sqrt{2}$.

The target qubit is unchanged (it's an eigenstate), but the control qubit acquires the eigenvalue phase. This is how QPE encodes the phase into a register that the inverse QFT can then read out.

In [7]:
%%
// Demonstrate phase kickback.
// The T gate has eigenvalue e^{i*pi/4} on eigenstate |1>.
// We use a controlled-T with the control in |+> and target in |1>.
// Expected: control acquires phase pi/4, target stays |1>.

cKick, err := builder.New("phase-kickback", 2).
	X(1).         // target = |1> (eigenstate of T)
	H(0).         // control = |+>
	Ctrl(gate.T, []int{0}, 1). // controlled-T
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Phase kickback circuit:")
gonbui.DisplayHTML(draw.SVG(cKick))

sim := statevector.New(2)
if err := sim.Evolve(cKick); err != nil {
	panic(err)
}
sv := sim.StateVector()

fmt.Println("Statevector after phase kickback:")
for i, amp := range sv {
	if cmplx.Abs(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %+.4f %+.4fi  (phase = %.4f rad)\n",
			i, real(amp), imag(amp), cmplx.Phase(amp))
	}
}

fmt.Println("\nInterpretation:")
fmt.Println("  Target qubit remains |1> in both terms — it is unchanged.")
fmt.Printf("  Control qubit: |01> has phase 0, |11> has phase pi/4 = %.4f rad\n", math.Pi/4)
fmt.Println("  The eigenvalue phase has 'kicked back' onto the control qubit.")
fmt.Println("  This is the core mechanism of Quantum Phase Estimation.")

Phase kickback circuit:
Statevector after phase kickback:
  |10> : +0.7071 +0.0000i  (phase = 0.0000 rad)
  |11> : +0.5000 +0.5000i  (phase = 0.7854 rad)

Interpretation:
  Target qubit remains |1> in both terms — it is unchanged.
  Control qubit: |01> has phase 0, |11> has phase pi/4 = 0.7854 rad
  The eigenvalue phase has 'kicked back' onto the control qubit.
  This is the core mechanism of Quantum Phase Estimation.


q0 
 
 q1 
 
 X 
 H 
 
 
 T

## Composing Circuits

Goqu provides several utilities for building complex circuits from simpler parts:

| Function | Description |
|:---|:---|
| `ir.Inverse(c)` | Reverses gate order and adjoints each gate (the "dagger" operation) |
| `ir.Repeat(c, n)` | Concatenates `c`'s operations `n` times |
| `builder.Compose(c, qubitMap)` | Appends circuit `c` into the builder with qubit remapping |
| `builder.ComposeInverse(c, qubitMap)` | Appends the inverse of `c` into the builder |

These are essential tools for building algorithms that reuse subcircuits — like applying QFT and then inverse QFT, or repeating a unitary $U^{2^k}$ times in QPE.

In [8]:
%%
// Demonstrate ir.Inverse: build QFT then invert it
qftCirc, _ := qpe.QFT(3)
qftInv := ir.Inverse(qftCirc)

fmt.Println("QFT circuit:")
gonbui.DisplayHTML(draw.SVG(qftCirc))
fmt.Println("ir.Inverse(QFT) — the adjoint:")
gonbui.DisplayHTML(draw.SVG(qftInv))

// Demonstrate ir.Repeat: repeat a small circuit 3 times
smallCirc, _ := builder.New("small", 1).T(0).Build()
repeated, err := ir.Repeat(smallCirc, 3)
if err != nil {
	panic(err)
}
fmt.Println("T gate repeated 3 times:")
gonbui.DisplayHTML(draw.SVG(repeated))

// Demonstrate Compose and ComposeInverse
b := builder.New("round-trip", 3)
b.X(0).X(2) // Prepare |101>
b.Compose(qftCirc, nil)              // Apply QFT
b.ComposeInverse(qftCirc, nil)       // Apply QFT-dagger (= inverse QFT)
roundTrip, _ := b.Build()

sim := statevector.New(3)
sim.Evolve(roundTrip)
sv := sim.StateVector()

fmt.Println("Compose QFT then ComposeInverse QFT on |101>:")
for i, amp := range sv {
	if cmplx.Abs(amp) > 1e-10 {
		fmt.Printf("  |%03b> : %.4f\n", i, amp)
	}
}
fmt.Println("Result: |101> recovered — the two operations cancel.")

QFT circuit:
ir.Inverse(QFT) — the adjoint:
T gate repeated 3 times:
Compose QFT then ComposeInverse QFT on |101>:
  |101> : (1.0000+0.0000i)
Result: |101> recovered — the two operations cancel.


q0 
 
 q1 
 
 q2 
 
 H 
 
 
 P(pi/2) 
 
 
 P(pi/4) 
 H 
 
 
 P(pi/2) 
 H

q0 
 
 q1 
 
 q2 
 
 
 
 
 H 
 
 
 P(-pi/2) 
 H 
 
 
 P(-pi/4) 
 
 
 P(-pi/2) 
 H

q0 
 
 T 
 T 
 T

## Predict, Then Verify

**Question:** What does QFT|000> produce? What will the measurement distribution look like?

**Pause and predict** before running the next cell.

*Hint: Think about the QFT formula with $j = 0$. What happens to the phase factor $e^{2\pi i \cdot 0 \cdot k / N}$?*

In [9]:
%%
// Verify: QFT|000> should give uniform superposition
qftCirc, _ := qpe.QFT(3)

sim := statevector.New(3)
prep, _ := builder.New("zero", 3).Build()
sim.Evolve(prep)
sim.Apply(qftCirc)
sv := sim.StateVector()

fmt.Println("QFT|000> statevector:")
allEqual := true
expectedAmp := 1.0 / math.Sqrt(8.0)
for k, amp := range sv {
	fmt.Printf("  |%03b> : %.6f  (expected: %.6f)\n", k, real(amp), expectedAmp)
	if cmplx.Abs(amp-complex(expectedAmp, 0)) > 1e-10 {
		allEqual = false
	}
}
fmt.Printf("\nAll amplitudes equal to 1/sqrt(8): %v\n", allEqual)

// Measure to show uniform distribution
bMeasure := builder.New("qft-measure", 3)
qpe.ApplyQFT(bMeasure, 3)
bMeasure.MeasureAll()
cMeasure, _ := bMeasure.Build()

simM := statevector.New(3)
counts, _ := simM.Run(cMeasure, 8000)

fmt.Println("\nMeasurement results (8000 shots, expect ~1000 each):")
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d (%.1f%%)\n", outcome, count, float64(count)/80.0)
}

svgHist := viz.Histogram(counts, viz.WithTitle("QFT|000> Measurement Distribution"))
gonbui.DisplayHTML(svgHist)

QFT|000> statevector:
  |000> : 0.353553  (expected: 0.353553)
  |001> : 0.353553  (expected: 0.353553)
  |010> : 0.353553  (expected: 0.353553)
  |011> : 0.353553  (expected: 0.353553)
  |100> : 0.353553  (expected: 0.353553)
  |101> : 0.353553  (expected: 0.353553)
  |110> : 0.353553  (expected: 0.353553)
  |111> : 0.353553  (expected: 0.353553)

All amplitudes equal to 1/sqrt(8): true

Measurement results (8000 shots, expect ~1000 each):
  |110> : 1031 (12.9%)
  |111> : 952 (11.9%)
  |000> : 1031 (12.9%)
  |010> : 998 (12.5%)
  |100> : 1025 (12.8%)
  |001> : 1059 (13.2%)
  |011> : 950 (11.9%)
  |101> : 954 (11.9%)


QFT|000> Measurement Distribution 
 
 
 
 0 
 
 200 
 
 400 
 
 600 
 
 800 
 
 1000 
 
 1200 
 
 000 
 
 001 
 
 010 
 
 011 
 
 100 
 
 101 
 
 110 
 
 111

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. What does the QFT do to the computational basis state |0...0⟩?
2. Why is the inverse QFT used in QPE rather than the forward QFT?
3. How many controlled-phase gates are needed for an n-qubit QFT, and why is this still exponentially better than the classical FFT?

## Exercises

### Exercise 1 — Build a 4-Qubit QFT Manually

Extend the manual construction pattern to 4 qubits. You will need:
- Qubit 0: H, CP($\pi/2$) with q1, CP($\pi/4$) with q2, CP($\pi/8$) with q3
- Qubit 1: H, CP($\pi/2$) with q2, CP($\pi/4$) with q3
- Qubit 2: H, CP($\pi/2$) with q3
- Qubit 3: H
- SWAPs: (0,3) and (1,2)

Verify your circuit matches `qpe.QFT(4)` by comparing statevectors on an input state.

In [10]:
%%
// Exercise 1: Build 4-qubit QFT manually
//
// TODO: Extend the manual QFT pattern to 4 qubits:
//   - Qubit 0: H, CP(pi/2) with q1, CP(pi/4) with q2, CP(pi/8) with q3
//   - Qubit 1: H, CP(pi/2) with q2, CP(pi/4) with q3
//   - Qubit 2: H, CP(pi/2) with q3
//   - Qubit 3: H
//   - SWAPs: (0,3) and (1,2)
// Then compare your circuit with qpe.QFT(4) on a test input.
//
// Hint: Use b.Apply(gate.CP(angle), control, target) for controlled-phase gates

// b := builder.New("qft4-manual", 4)
//
// // TODO: Qubit 0 -- H and three controlled-phase gates
//
// // TODO: Qubit 1 -- H and two controlled-phase gates
//
// // TODO: Qubit 2 -- H and one controlled-phase gate
//
// // TODO: Qubit 3 -- H only
//
// // TODO: SWAPs to reverse qubit order
//
// manual4, err := b.Build()
// if err != nil {
// 	panic(err)
// }
//
// fmt.Println("Manual 4-qubit QFT:")
// fmt.Println(draw.String(manual4))
//
// // Compare with built-in on |1010>
// builtin4, _ := qpe.QFT(4)
// prep, _ := builder.New("prep", 4).X(1).X(3).Build()
//
// sim1 := statevector.New(4)
// sim1.Evolve(prep)
// sim1.Apply(manual4)
// sv1 := sim1.StateVector()
//
// sim2 := statevector.New(4)
// sim2.Evolve(prep)
// sim2.Apply(builtin4)
// sv2 := sim2.StateVector()
//
// match := true
// for i := range sv1 {
// 	if cmplx.Abs(sv1[i]-sv2[i]) > 1e-10 {
// 		match = false
// 		break
// 	}
// }
// fmt.Printf("\nManual 4-qubit QFT matches built-in on |1010>: %v\n", match)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


### Exercise 2 — Verify QFT/Inverse-QFT Identity on a Superposition State

Create a non-trivial 3-qubit state by applying some gates (e.g., H on qubit 0, CNOT(0,1), T on qubit 2), then apply QFT followed by inverse QFT. Verify the original statevector is recovered.

In [11]:
%%
// Exercise 2: QFT -> inverse QFT is identity on an arbitrary state
//
// TODO: Create a non-trivial 3-qubit state, apply QFT then inverse QFT,
// and verify the original statevector is recovered.
//
// Steps:
//   1. Prepare a state: e.g., H(0), CNOT(0,1), T(2)
//   2. Record the statevector before the round-trip
//   3. Apply qpe.QFT(3) then qpe.InverseQFT(3)
//   4. Compare the final statevector with the original
//
// Hint: Use sim.Apply(circuit) to apply a pre-built circuit to the simulator

// prep, _ := builder.New("arbitrary", 3).
// 	H(0).
// 	CNOT(0, 1).
// 	T(2).
// 	Build()
//
// sim := statevector.New(3)
// sim.Evolve(prep)
// svBefore := sim.StateVector()
//
// // TODO: Build QFT and inverse QFT circuits
// // qftCirc, _ := qpe.QFT(3)
// // invCirc, _ := qpe.InverseQFT(3)
//
// // TODO: Apply QFT then inverse QFT
// // sim.Apply(qftCirc)
// // sim.Apply(invCirc)
// // svAfter := sim.StateVector()
//
// // TODO: Compare svBefore and svAfter
// // match := true
// // for i := range svBefore {
// // 	if cmplx.Abs(svBefore[i]-svAfter[i]) > 1e-10 {
// // 		match = false
// // 		break
// // 	}
// // }
// // fmt.Printf("Original state recovered: %v\n", match)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


## Key Takeaways

1. **The QFT** maps computational basis states to the frequency basis: $\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_k e^{2\pi i jk/N} |k\rangle$. It uses $O(n^2)$ gates versus the classical FFT's $O(N \log N)$ — an exponential improvement.

2. **Circuit structure:** H gates create superposition on each qubit; controlled-phase gates $\text{CP}(\pi/2^{j-i})$ encode the frequency relationships; final SWAPs fix the output ordering.

3. **The inverse QFT** reverses the transformation. QFT followed by inverse QFT is the identity on any state.

4. **QFT on $|0\rangle$** produces a uniform superposition. QFT on $|j\rangle$ produces uniform amplitudes with phases that increase at frequency $j$.

5. **Phase kickback** transfers an eigenvalue phase from a controlled unitary onto the control qubit. This is how QPE encodes phases into a register for the inverse QFT to decode.

6. **Circuit composition** tools (`ir.Inverse`, `ir.Repeat`, `Compose`, `ComposeInverse`) make it easy to build complex algorithms from reusable subcircuits.

---

**Next up:** Notebook 08 — Quantum Phase Estimation, where we combine the QFT, phase kickback, and controlled unitaries into a complete algorithm for extracting eigenvalue phases.